# Data 620 Week 8 Assignment - Project 3
### Author: Pedro P Gatica Jr

In [1]:
import pandas as pd
import csv
import os
from datetime import datetime
import random
import nltk
from nltk.corpus import names
from collections import Counter

nltk.download('names')

[nltk_data] Downloading package names to
[nltk_data]     /Users/Audiorunner13/nltk_data...
[nltk_data]   Package names is already up-to-date!


True

In [2]:
RESULTS_FILE = "project3_results.csv"
results = []   # in-memory log of every experiment

In [3]:
labeled_names = ([(name, 'male') for name in names.words('male.txt')] +
                 [(name, 'female') for name in names.words('female.txt')])

In [4]:
random.seed(42)
random.shuffle(labeled_names)

test_names    = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names   = labeled_names[1000:]

print(len(train_names), len(devtest_names), len(test_names))

6944 500 500


In [5]:
def run_experiment(version, description, feature_fn):
    """Train on train_names, evaluate on dev-test, and log the result."""
    train_set   = [(feature_fn(n), g) for (n, g) in train_names]
    devtest_set = [(feature_fn(n), g) for (n, g) in devtest_names]

    clf = nltk.NaiveBayesClassifier.train(train_set)
    acc = nltk.classify.accuracy(clf, devtest_set)

    # Compare against the previous version, if there is one
    change = acc - results[-1]["devtest_accuracy"] if results else 0.0

    results.append({
        "version": version,
        "description": description,
        "devtest_accuracy": round(acc, 4),
        "change_from_prev": round(change, 4),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })
    save_results()

    print(f"{version}: {acc:.4f} ({change:+.4f})  {description}")
    return clf

In [6]:
def save_results():
    """Overwrite the CSV with the full results log."""
    with open(RESULTS_FILE, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

In [7]:
# v1: baseline from the chapter
def features_v1(name):
    return {'last_letter': name[-1].lower()}

In [8]:
# v2: add first letter
def features_v2(name):
    f = features_v1(name)
    f['first_letter'] = name[0].lower()
    return f

In [9]:
# v3: add last two letters
def features_v3(name):
    f = features_v2(name)
    f['suffix2'] = name[-2:].lower()
    return f

In [10]:
# v4: add last three letters and length
def features_v4(name):
    f = features_v3(name)
    f['suffix3'] = name[-3:].lower()
    f['length'] = len(name)
    return f

In [11]:
# v5: add letter-presence features
def features_v5(name):
    f = features_v4(name)
    for letter in 'abcdefghijklmnopqrstuvwxyz':
        f[f'has({letter})'] = (letter in name.lower())
    return f

In [32]:
def features_v6d(name):
    f = features_v5(name)
    n = name.lower()
    f['penultimate'] = n[-2] if len(n) > 1 else ''
    # any doubled letter in the last 4 letters (Benny, Kenny, Abby, Tabby)
    tail = n[-4:]
    f['double_in_tail'] = any(tail[i] == tail[i+1] for i in range(len(tail) - 1))
    # consonant + y/ie nickname shape
    f['nick_ending'] = n.endswith(('y', 'ie')) and len(n) <= 6
    return f

In [37]:
def run_experiment_clf(version, description, feature_fn, trainer):
    """Like run_experiment, but takes any NLTK trainer and logs to the same CSV."""
    train_set   = [(feature_fn(n), g) for (n, g) in train_names]
    devtest_set = [(feature_fn(n), g) for (n, g) in devtest_names]

    clf = trainer(train_set)
    acc = nltk.classify.accuracy(clf, devtest_set)
    change = acc - results[-1]["devtest_accuracy"] if results else 0.0

    results.append({
        "version": version,
        "description": description,
        "devtest_accuracy": round(acc, 4),
        "change_from_prev": round(change, 4),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })
    save_results()

    print(f"{version}: {acc:.4f} ({change:+.4f})  {description}")
    return clf

In [12]:
run_experiment("v1", "last letter (baseline)",         features_v1)

v1: 0.7540 (+0.0000)  last letter (baseline)


In [13]:
run_experiment("v2", "+ first letter",                 features_v2)

v2: 0.7540 (+0.0000)  + first letter


In [14]:
run_experiment("v3", "+ last two letters",             features_v3)

v3: 0.7820 (+0.0280)  + last two letters


In [15]:
run_experiment("v4", "+ last three letters, length",   features_v4)

v4: 0.7920 (+0.0100)  + last three letters, length


In [16]:
run_experiment("v5", "+ letter presence (a-z)",        features_v5)

v5: 0.8060 (+0.0140)  + letter presence (a-z)


In [33]:
run_experiment("v6d", "+ penultimate, double letter in tail, short -y/-ie", features_v6d)

v6d: 0.8020 (-0.0040)  + penultimate, double letter in tail, short -y/-ie


In [34]:
df = pd.read_csv(RESULTS_FILE)
print(df.to_string(index=False))

version                                        description  devtest_accuracy  change_from_prev           timestamp
     v1                             last letter (baseline)             0.754             0.000 2026-11-03 18:17:58
     v2                                     + first letter             0.754             0.000 2026-11-03 18:17:58
     v3                                 + last two letters             0.782             0.028 2026-11-03 18:17:58
     v4                       + last three letters, length             0.792             0.010 2026-11-03 18:17:58
     v5                            + letter presence (a-z)             0.806             0.014 2026-11-03 18:17:58
    v6d + penultimate, double letter in tail, short -y/-ie             0.802            -0.004 2026-11-03 19:27:10


In [18]:
best_features = features_v5   # change to whichever version won

In [19]:
train_set = [(best_features(n), g) for (n, g) in train_names]
test_set  = [(best_features(n), g) for (n, g) in test_names]
final_clf = nltk.NaiveBayesClassifier.train(train_set)

In [20]:
test_acc = nltk.classify.accuracy(final_clf, test_set)
best_devtest = max(r["devtest_accuracy"] for r in results)

In [21]:
print(f"Dev-test accuracy: {best_devtest:.4f}")
print(f"Test accuracy:     {test_acc:.4f}")
print(f"Difference:        {test_acc - best_devtest:+.4f}")

Dev-test accuracy: 0.8060
Test accuracy:     0.7840
Difference:        -0.0220


In [22]:
# Save the final result alongside your log
with open("project3_final.txt", "w") as f:
    f.write(f"Best dev-test: {best_devtest:.4f}\nTest: {test_acc:.4f}\n")

In [23]:
# Retrain the v5 classifier (or use: clf_v5 = run_experiment(...) if you saved it)
train_set = [(features_v5(n), g) for (n, g) in train_names]
clf_v5 = nltk.NaiveBayesClassifier.train(train_set)

In [24]:
# Collect dev-test errors
errors = []
for (name, tag) in devtest_names:
    guess = clf_v5.classify(features_v5(name))
    if guess != tag:
        errors.append((tag, guess, name))

print(f"Errors: {len(errors)} of {len(devtest_names)}")

Errors: 97 of 500


### Confusion matrix: which direction are the mistakes?

In [25]:
confusion = Counter((tag, guess) for (tag, guess, name) in errors)
print("male   -> predicted female:", confusion[('male', 'female')])
print("female -> predicted male:  ", confusion[('female', 'male')])

male   -> predicted female: 51
female -> predicted male:   46


### Which endings cause the most errors?

In [26]:
print("\nMost common last letters among errors:")
for letter, count in Counter(n[-1].lower() for (_, _, n) in errors).most_common(8):
    print(f"  -{letter}: {count}")


Most common last letters among errors:
  -e: 25
  -y: 19
  -n: 13
  -l: 9
  -t: 6
  -s: 5
  -d: 4
  -i: 4


In [27]:
print("\nMost common last-two-letter endings among errors:")
for suf, count in Counter(n[-2:].lower() for (_, _, n) in errors).most_common(10):
    print(f"  -{suf}: {count}")


Most common last-two-letter endings among errors:
  -ie: 9
  -ny: 6
  -en: 5
  -ne: 4
  -in: 4
  -ll: 3
  -dy: 3
  -ey: 2
  -lo: 2
  -rd: 2


### Compare error rate per ending against how common the ending is. 
##### An ending with many errors may simply be frequent, so this normalizes it:

In [28]:
suffix_total = Counter(n[-2:].lower() for (n, _) in devtest_names)
suffix_err   = Counter(n[-2:].lower() for (_, _, n) in errors)

print("\nSuffixes with the highest error rate (at least 5 names):")


Suffixes with the highest error rate (at least 5 names):


In [29]:
rows = [(s, suffix_err[s], suffix_total[s], suffix_err[s] / suffix_total[s])
        for s in suffix_total if suffix_total[s] >= 5]
for s, e, t, r in sorted(rows, key=lambda x: -x[3])[:10]:
    print(f"  -{s}: {e}/{t} wrong ({r:.0%})")

  -ny: 6/8 wrong (75%)
  -ll: 3/5 wrong (60%)
  -en: 5/11 wrong (45%)
  -is: 2/6 wrong (33%)
  -ie: 9/28 wrong (32%)
  -rd: 2/7 wrong (29%)
  -el: 2/7 wrong (29%)
  -ey: 2/8 wrong (25%)
  -in: 4/16 wrong (25%)
  -th: 2/9 wrong (22%)


### Read the actual mistakes. 
##### This is the step that tends to give you feature ideas:

In [30]:
print("\nMale names predicted female:")
print(sorted(n for (t, g, n) in errors if t == 'male'))

print("\nFemale names predicted male:")
print(sorted(n for (t, g, n) in errors if t == 'female'))


Male names predicted female:
['Abby', 'Aldis', 'Allen', 'Alley', 'Alphonse', 'Andre', 'Antony', 'Bailie', 'Barri', 'Benjamen', 'Benji', 'Benny', 'Boyce', 'Brandy', 'Claire', 'Costa', 'Daryl', 'Davide', 'Erl', 'Etienne', 'Georgie', 'Germaine', 'Jessee', 'Kenny', 'Kermie', 'Kingsly', 'Manny', 'Marlowe', 'Martainn', 'Melvyn', 'Meredith', 'Merlin', 'Merry', 'Nathaniel', 'Normie', 'Ole', 'Oral', 'Richie', 'Roddie', 'Rodrique', 'Seth', 'Sollie', 'Tabby', 'Tanny', 'Thaine', 'Tuckie', 'Tye', 'Vinny', 'Wayne', 'Yehudi', 'Zacharie']

Female names predicted male:
['Beatriz', 'Bliss', 'Brigid', 'Brooks', 'Cherey', 'Cordy', 'Dallas', 'Doralin', 'Dory', 'Eden', 'Eleanor', 'Gillan', 'Ginnifer', 'Glad', 'Gretchen', 'Grissel', 'Harriott', 'Hildagard', 'Ivett', 'Ivy', 'Joell', 'Lil', 'Lindsay', 'Lust', 'Margalo', 'Marlo', 'Mei', 'Melicent', 'Mellicent', 'Midge', 'Moll', 'Pris', 'Raf', 'Raynell', 'Robbin', 'Roselin', 'Shannen', 'Sharon', 'Sher', 'Stoddard', 'Tamar', 'Trish', 'Trudy', 'Ulrike', 'Vikky', 

In [31]:
totals = Counter(tag for (_, tag) in devtest_names)
print("Dev-test class counts:", dict(totals))
print(f"Male error rate:   {51 / totals['male']:.1%}")
print(f"Female error rate: {46 / totals['female']:.1%}")

Dev-test class counts: {'female': 297, 'male': 203}
Male error rate:   25.1%
Female error rate: 15.5%


In [36]:
# MaxEnt (set max_iter low so it finishes quickly)
run_experiment_clf("v7-maxent", "MaxEnt on v6d features", features_v6d,
                   lambda ts: nltk.MaxentClassifier.train(ts, max_iter=15, trace=0))

v7-maxent: 0.8040  MaxEnt on v6d features


<ConditionalExponentialClassifier: 2 labels, 2433 features>

In [38]:
maxent_trainer = lambda iters: (lambda ts: nltk.MaxentClassifier.train(
    ts, algorithm='GIS', max_iter=iters, trace=0))

In [39]:
clf_me15 = run_experiment_clf("v7a", "MaxEnt (GIS, 15 iter) on v5 features",
                              features_v5, maxent_trainer(15))

v7a: 0.7940 (-0.0080)  MaxEnt (GIS, 15 iter) on v5 features


In [41]:
clf_me40 = run_experiment_clf("v7b", "MaxEnt (GIS, 40 iter) on v5 features",
                              features_v5, maxent_trainer(40))

v7b: 0.8000 (+0.0000)  MaxEnt (GIS, 40 iter) on v5 features
